In [3]:
import numpy as np
from scipy.optimize import minimize

# ----------------------------
# INPUT: expected returns
# ----------------------------
assets = [
    "Scoria Paste",
    "Thermalite",
    "Sulfur Ltd",
    "Obsidian",
    "Magma Ink",
    "Incense",
]

alpha = np.array([3.0, 2.5, 2.2, 1.8, 1.2, 1.0])

# Normalize alphas (important for stability)
alpha = alpha / np.max(np.abs(alpha))


# ----------------------------
# PARAMETERS
# ----------------------------
BUDGET_EXPONENT = 1_000_000  # from problem


# ----------------------------
# FEE FUNCTION
# ----------------------------
def fee(w):
    # w is fraction of total budget (e.g. 0.1 = 10%)
    v = np.abs(w)

    # avoid numerical underflow
    scaled = v / 100.0

    # compute safely using logs
    # fee = scaled^(1 + BUDGET_EXPONENT)
    with np.errstate(divide='ignore'):
        log_fee = (1 + BUDGET_EXPONENT) * np.log(scaled + 1e-18)
    
    return np.exp(log_fee)


# ----------------------------
# OBJECTIVE (maximize PnL)
# ----------------------------
def objective(w):
    pnl = np.sum(w * alpha)
    total_fee = np.sum(fee(w))
    return -(pnl - total_fee)  # minimize negative


# ----------------------------
# CONSTRAINTS
# ----------------------------

# Total absolute allocation <= 1
def budget_constraint(w):
    return 1.0 - np.sum(np.abs(w))


constraints = [
    {"type": "ineq", "fun": budget_constraint}
]

# bounds: allow long and short
bounds = [(-1, 1) for _ in range(len(alpha))]


# ----------------------------
# INITIAL GUESS
# ----------------------------
w0 = np.zeros_like(alpha)


# ----------------------------
# OPTIMIZATION
# ----------------------------
result = minimize(
    objective,
    w0,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"maxiter": 1000, "ftol": 1e-9}
)

# ----------------------------
# OUTPUT
# ----------------------------
w_opt = result.x

print("Optimal weights:")
for name, w in zip(assets, w_opt):
    print(f"{name:15s}: {w*100:6.2f}%")

print("\nTotal allocation:", np.sum(np.abs(w_opt)))
print("Expected PnL:", np.sum(w_opt * alpha) - np.sum(fee(w_opt)))

Optimal weights:
Scoria Paste   : 100.00%
Thermalite     :   0.00%
Sulfur Ltd     :  -0.00%
Obsidian       :  -0.00%
Magma Ink      :  -0.00%
Incense        :  -0.00%

Total allocation: 0.999999998740893
Expected PnL: 0.9999999754141762
